In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# DNN Model
model = Sequential([
    Dense(32, activation='relu', input_shape=(1,)),
    Dense(64, activation='relu'),
    Dense(128, activation='relu'),
    Dense(1)
])

model.summary()

In [ ]:
# Generate samples
x = np.linspace(-20, 20, 1000)

# Polynomial
y = x**3 - 2*x**2 + 5*x + 1

# Normalize to [-1,1]
x_norm = x / 20

# Normalize output
y_max = np.max(np.abs(y))
y_norm = y / y_max

df = pd.DataFrame({
    "Original x": x,
    "Normalized x": x_norm,
    "Original y": y,
    "Normalized y": y_norm
})

print(df.head(20))

In [ ]:
# 10% Test
X_train, X_test, y_train, y_test = train_test_split(
    x_norm, y_norm,
    test_size=0.10,
    random_state=42
)

# 10% Validation from remaining data
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1111,   # Gives ~10% of total data
    random_state=42
)

# Compile model
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# Early Stopping Callback
early_stop = EarlyStopping(
    monitor='val_loss',       # Monitor validation loss
    patience=5,              # Stop after 10 epochs without improvement
    restore_best_weights=True,# Restore best model weights
    verbose=1
)

# Train Model
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,               # Set a higher maximum
    batch_size=32,
    callbacks=[early_stop]
)

# Test Evaluation
loss, mae = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test MAE :", mae)

In [ ]:
# Accuracy (MAE)
plt.figure(figsize=(8,5))
plt.plot(history.history['mae'], label='Training Accuracy (MAE)')
plt.plot(history.history['val_mae'], label='Validation Accuracy (MAE)')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Error (Loss)
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training vs Validation Error')
plt.legend()
plt.grid(True)
plt.show()